# Proyecto Data Science - Cuenca Cuyana

## Auditoría y selección de datasets

En este notebook realizaremos una primera exploración de los
datasets disponibles para evaluar cuáles resultan adecuados para
estudiar la dinámica productiva de hidrocarburos en la Cuenca Cuyana.

# Tema general: Análisis de la dinámica productiva de hidrocarburos en la Cuenca Cuyana.
Línea 1: evolución/resiliencia productiva de Cuyana.
Línea 2: tiempos de maduración o ramp-up de sus pozos.

## Preguntas de trabajo

Antes de definir el modelo final necesitamos comprobar:

1. ¿Los datasets contienen suficiente información de la Cuenca Cuyana?
2. ¿Podemos seguir un mismo pozo a través del tiempo?
3. ¿Qué período temporal tenemos disponible?
4. ¿Contamos con una fecha confiable de primera producción?
5. ¿Podemos reconstruir la evolución mensual de petróleo y gas?
6. ¿Cuántos pozos tienen suficiente historial para estudiar
   el tiempo hasta alcanzar su pico de producción?

## Alcance de esta primera auditoría

Esta notebook no busca realizar todavía el análisis exploratorio completo
ni construir modelos predictivos.

Su objetivo es evaluar la calidad y disponibilidad de los datos, construir
una base histórica consistente de la Cuenca Cuyana y determinar qué líneas
de investigación son viables antes de definir el problema final del proyecto.

Las conclusiones obtenidas son preliminares y serán complementadas con el
análisis de los datasets de contexto macroeconómico y operativo.

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Definición de rutas

Definimos las rutas de las carpetas que contienen los distintos grupos
de datasets del proyecto para facilitar su carga y organización.

In [71]:
RUTA_MICRO = "../data/raw/01_micro_datos_pozos/"
RUTA_PADRONES = "../data/raw/02_padrones/"
RUTA_MACRO = "../data/raw/03_contexto_macro/"
RUTA_OPCIONALES = "../data/raw/04_opcionales/"

## Exploración inicial de los microdatos de producción

Antes de combinar todos los años, analizaremos el dataset correspondiente
a 2025 para conocer su estructura, variables y calidad de los datos.

Si la estructura resulta adecuada, posteriormente verificaremos que sea
compatible con los demás años.

In [72]:
# Se utiliza 2025 como año de referencia para validar
# la estructura antes de integrar todos los archivos.
df_2025 = pd.read_csv(
    RUTA_MICRO + "produccin-de-pozos-de-gas-y-petrleo-2025.csv"
)

C:\Users\milib\AppData\Local\Temp\ipykernel_20620\3610457760.py:3: DtypeWarning: Columns (0: sub_tipo_recurso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2025 = pd.read_csv(


In [19]:
print("Filas:", df_2025.shape[0])
print("Columnas:", df_2025.shape[1])

df_2025.head()

Filas: 991844
Columnas: 38


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,...,idareayacimiento,areayacimiento,cuenca,provincia,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data
0,Z001,2025,1,145626,0.0,0.0,0.0,0.0,0.0,0.0,...,AGR,GENERAL ROCA,NEUQUINA,Rio Negro,CONVENCIONAL,Sin Proyecto,EXPLORACION,EXPLORACION,NaN,2025-01-31
1,Z001,2025,1,145620,0.0,0.0,0.0,0.0,0.0,0.0,...,FDRO,FLOR DE ROCA,NEUQUINA,Rio Negro,CONVENCIONAL,Sin Proyecto,EXPLOTACION,AVANZADA,NaN,2025-01-31
2,Z001,2025,1,32189,0.0,0.0,0.0,0.0,0.0,0.0,...,ABO,"BLANCO DE LOS OLIVOS-BLOQUE ""A""",NEUQUINA,Rio Negro,CONVENCIONAL,Sin Proyecto,EXPLORACION,EXPLORACION,NaN,2025-01-31
3,Z001,2025,1,145604,0.0,0.0,0.0,0.0,0.0,0.0,...,FDRO,FLOR DE ROCA,NEUQUINA,Rio Negro,CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,NaN,2025-01-31
4,Z001,2025,1,145615,0.0,0.0,0.0,0.0,0.0,0.0,...,FDRO,FLOR DE ROCA,NEUQUINA,Rio Negro,CONVENCIONAL,Sin Proyecto,SERVICIO,SUMIDERO,NaN,2025-01-31


In [20]:
df_2025.info()

<class 'pandas.DataFrame'>
RangeIndex: 991844 entries, 0 to 991843
Data columns (total 38 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   idempresa               991844 non-null  str    
 1   anio                    991844 non-null  int64  
 2   mes                     991844 non-null  int64  
 3   idpozo                  991844 non-null  int64  
 4   prod_pet                991844 non-null  float64
 5   prod_gas                991844 non-null  float64
 6   prod_agua               991844 non-null  float64
 7   iny_agua                991844 non-null  float64
 8   iny_gas                 991844 non-null  float64
 9   iny_co2                 991844 non-null  float64
 10  iny_otro                991844 non-null  float64
 11  tef                     991844 non-null  float64
 12  vida_util               72154 non-null   float64
 13  tipoextraccion          991811 non-null  str    
 14  tipoestado              991811 

In [21]:
df_2025.describe().T

,count,mean,std,min,25%,50%,75%,max
anio,991844.0,2025.000000,0.000000,2025.00,2025.0,2025.0,2025.000000,2.025000e+03
mes,991844.0,6.504990,3.452205,1.00,4.0,7.0,10.000000,1.200000e+01
idpozo,991844.0,103275.406626,42831.964464,212.00,71544.0,110837.0,133743.000000,1.671530e+05
prod_pet,991844.0,47.006464,311.885523,-0.08,0.0,0.0,6.572552,2.659326e+04
prod_gas,991844.0,52.018704,880.462773,-12.14,0.0,0.0,0.000000,1.335200e+05
prod_agua,991844.0,333.460530,1146.830277,-12.11,0.0,0.0,10.833122,3.777149e+04
iny_agua,991844.0,336.201872,1995.134353,0.00,0.0,0.0,0.000000,2.188378e+05
iny_gas,991844.0,35.081000,11882.009742,0.00,0.0,0.0,0.000000,9.267366e+06
iny_co2,991844.0,0.000000,0.000000,0.00,0.0,0.0,0.000000,0.000000e+00
iny_otro,991844.0,1.443327,216.565422,0.00,0.0,0.0,0.000000,4.739675e+04


In [22]:
df_2025.columns.tolist()

['idempresa',
 'anio',
 'mes',
 'idpozo',
 'prod_pet',
 'prod_gas',
 'prod_agua',
 'iny_agua',
 'iny_gas',
 'iny_co2',
 'iny_otro',
 'tef',
 'vida_util',
 'tipoextraccion',
 'tipoestado',
 'tipopozo',
 'observaciones',
 'fechaingreso',
 'rectificado',
 'habilitado',
 'idusuario',
 'empresa',
 'sigla',
 'formprod',
 'profundidad',
 'formacion',
 'idareapermisoconcesion',
 'areapermisoconcesion',
 'idareayacimiento',
 'areayacimiento',
 'cuenca',
 'provincia',
 'tipo_de_recurso',
 'proyecto',
 'clasificacion',
 'subclasificacion',
 'sub_tipo_recurso',
 'fecha_data']

### Observación inicial

El dataset de producción correspondiente a 2025 contiene 991.844 registros
y 38 variables. Cada registro incluye información temporal, productiva y
características asociadas a los pozos.

El volumen del archivo es considerable, por lo que antes de combinar los
distintos años se analizará su estructura y se filtrarán los registros
correspondientes a la Cuenca Cuyana.

In [23]:
df_2025["cuenca"].value_counts()

cuenca
GOLFO SAN JORGE    515227
NEUQUINA           383431
CUYANA              44163
AUSTRAL             37484
NOROESTE            11335
NORESTE               132
ÑIRIHUAU               24
CAÑADON ASFALTO        12
Name: count, dtype: int64

In [24]:
df_cuyana_2025 = df_2025[
    df_2025["cuenca"] == "CUYANA"
].copy()
print("Registros de Cuyana:", len(df_cuyana_2025))
print("Pozos únicos:", df_cuyana_2025["idpozo"].nunique())

Registros de Cuyana: 44163
Pozos únicos: 3685


### Foco en la Cuenca Cuyana

La variable `cuenca` permite identificar directamente los registros
correspondientes a la Cuenca Cuyana.

Durante 2025 se registran 44.163 observaciones asociadas a la cuenca,
correspondientes a 3.685 pozos únicos.

Esto confirma que el dataset contiene un volumen considerable de
información específica de la región de interés y permite trabajar con
la cuenca sin utilizar la provincia como variable sustituta.

In [25]:
faltantes = pd.DataFrame({
    "cantidad": df_cuyana_2025.isna().sum(),
    "porcentaje": (
        df_cuyana_2025.isna().mean() * 100
    ).round(2)
})

faltantes = faltantes[
    faltantes["cantidad"] > 0
].sort_values("porcentaje", ascending=False)

faltantes

,cantidad,porcentaje
sub_tipo_recurso,44163,100.00
vida_util,38771,87.79
observaciones,37559,85.05
clasificacion,5376,12.17
subclasificacion,5376,12.17


### Valores faltantes

La presencia de valores faltantes no es uniforme entre las variables.

`sub_tipo_recurso` presenta un 100% de valores faltantes dentro de los
registros de la Cuenca Cuyana en 2025, mientras que `vida_util` y
`observaciones` presentan porcentajes superiores al 85%.

En cambio, `clasificacion` y `subclasificacion` presentan aproximadamente
un 12% de valores faltantes.

Por el momento no se eliminarán ni imputarán valores. Primero se evaluará
la utilidad de cada variable y se comprobará si este comportamiento se
mantiene en los demás años.

In [26]:
df_cuyana_2025.duplicated().sum()

np.int64(0)

In [27]:
duplicados_pozo_mes = df_cuyana_2025.duplicated(
    subset=["idpozo", "anio", "mes"]
).sum()

duplicados_pozo_mes

np.int64(0)

### Duplicados y granularidad

No se encontraron filas completamente duplicadas en los registros de
la Cuenca Cuyana durante 2025.

Además, no existen duplicados para la combinación `idpozo`, `anio` y
`mes`. Esto indica que cada pozo presenta como máximo un registro por
mes dentro del período analizado.

Este resultado es especialmente importante para el proyecto, ya que
permite reconstruir posteriormente la evolución temporal de cada pozo
sin duplicidad mensual.

## Continuidad temporal de los pozos

Para estudiar la evolución productiva y posteriormente analizar el
ramp-up, es necesario conocer cuántos meses de información posee cada
pozo durante 2025.

In [28]:
meses_por_pozo = (
    df_cuyana_2025
    .groupby("idpozo")["mes"]
    .nunique()
)

meses_por_pozo.value_counts().sort_index()

mes
2        1
5        5
8        3
12    3676
Name: count, dtype: int64

### Interpretación de la continuidad temporal

La continuidad temporal de los registros durante 2025 es muy alta.

De los 3.685 pozos únicos identificados en la Cuenca Cuyana:

- 3.676 pozos presentan información para los 12 meses del año.
- 3 pozos presentan 8 meses de información.
- 5 pozos presentan 5 meses.
- 1 pozo presenta 2 meses.

Esto significa que aproximadamente el 99,76% de los pozos cuenta con
información para los 12 meses de 2025.

La alta continuidad mensual es favorable para reconstruir curvas de
producción y realizar análisis temporales a nivel de pozo.

Sin embargo, los pozos con menos de 12 meses no serán considerados
automáticamente como registros incompletos o erróneos, ya que podrían
corresponder a pozos que comenzaron o finalizaron su actividad durante
el año. Esta situación se verificará posteriormente utilizando la fecha
de primera producción y el estado de cada pozo.

## Compatibilidad entre los datasets anuales

Una vez validada la estructura del dataset 2025, se analizará si los
archivos correspondientes a 2021–2026 presentan una estructura
compatible.

El objetivo es determinar si pueden combinarse en un único dataset
histórico para la Cuenca Cuyana.

In [29]:
archivos_anuales = {
    2021: RUTA_MICRO + "produccin-de-pozos-de-gas-y-petrleo-2021.csv",
    2022: RUTA_MICRO + "produccin-de-pozos-de-gas-y-petrleo-2022.csv",
    2023: RUTA_MICRO + "produccin-de-pozos-de-gas-y-petrleo-2023.csv",
    2024: RUTA_MICRO + "produccin-de-pozos-de-gas-y-petrleo-2024.csv",
    2025: RUTA_MICRO + "produccin-de-pozos-de-gas-y-petrleo-2025.csv",
    2026: RUTA_MICRO + "produccin-de-pozos-de-gas-y-petrleo-2026-ddjj-abiertas-y-cerradas-.csv"
}

In [30]:
estructura_anual = {}
for anio, archivo in archivos_anuales.items():
    
    df_temp = pd.read_csv(archivo, nrows=0)
    
    estructura_anual[anio] = {
        "cantidad_columnas": len(df_temp.columns),
        "columnas": list(df_temp.columns)
    }

for anio, info in estructura_anual.items():
    print(
        f"{anio}: {info['cantidad_columnas']} columnas"
    )

2021: 38 columnas
2022: 38 columnas
2023: 38 columnas
2024: 39 columnas
2025: 38 columnas
2026: 38 columnas


In [31]:
columnas_referencia = estructura_anual[2025]["columnas"]

for anio, info in estructura_anual.items():
    mismas_columnas = info["columnas"] == columnas_referencia
    
    print(f"{anio}: mismas columnas que 2025 -> {mismas_columnas}")

2021: mismas columnas que 2025 -> True
2022: mismas columnas que 2025 -> True
2023: mismas columnas que 2025 -> True
2024: mismas columnas que 2025 -> False
2025: mismas columnas que 2025 -> True
2026: mismas columnas que 2025 -> True


In [32]:
columnas_2024 = estructura_anual[2024]["columnas"]
columnas_2025 = estructura_anual[2025]["columnas"]

print("Columnas que están en 2025 pero no en 2024:")
print(set(columnas_2025) - set(columnas_2024))

print("\nColumnas que están en 2024 pero no en 2025:")
print(set(columnas_2024) - set(columnas_2025))

Columnas que están en 2025 pero no en 2024:
set()

Columnas que están en 2024 pero no en 2025:
{'id'}


In [33]:
for i, (col_2024, col_2025) in enumerate(zip(columnas_2024, columnas_2025)):
    if col_2024 != col_2025:
        print(
            f"Posición {i}: "
            f"2024 = '{col_2024}' | "
            f"2025 = '{col_2025}'"
        )

In [34]:
print("2024:", columnas_2024)
print("\n2025:", columnas_2025)

2024: ['idempresa', 'anio', 'mes', 'idpozo', 'prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'iny_gas', 'iny_co2', 'iny_otro', 'tef', 'vida_util', 'tipoextraccion', 'tipoestado', 'tipopozo', 'observaciones', 'fechaingreso', 'rectificado', 'habilitado', 'idusuario', 'empresa', 'sigla', 'formprod', 'profundidad', 'formacion', 'idareapermisoconcesion', 'areapermisoconcesion', 'idareayacimiento', 'areayacimiento', 'cuenca', 'provincia', 'tipo_de_recurso', 'proyecto', 'clasificacion', 'subclasificacion', 'sub_tipo_recurso', 'fecha_data', 'id']

2025: ['idempresa', 'anio', 'mes', 'idpozo', 'prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'iny_gas', 'iny_co2', 'iny_otro', 'tef', 'vida_util', 'tipoextraccion', 'tipoestado', 'tipopozo', 'observaciones', 'fechaingreso', 'rectificado', 'habilitado', 'idusuario', 'empresa', 'sigla', 'formprod', 'profundidad', 'formacion', 'idareapermisoconcesion', 'areapermisoconcesion', 'idareayacimiento', 'areayacimiento', 'cuenca', 'provincia', 'tipo_de_recurs

In [35]:
id_2024 = pd.read_csv(
    archivos_anuales[2024],
    usecols=["id"]
)

id_2024.head(10)

,id
0,32173202401
1,145615202401
2,145623202401
3,145618202401
4,145625202401
5,145617202401
6,145622202401
7,145603202401
8,145620202401
9,32190202401


In [36]:
id_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 983551 entries, 0 to 983550
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   id      983551 non-null  int64
dtypes: int64(1)
memory usage: 7.5 MB


In [37]:
id_2024["id"].nunique()

983551

In [38]:
id_2024["id"].head(20)

0      32173202401
1     145615202401
2     145623202401
3     145618202401
4     145625202401
5     145617202401
6     145622202401
7     145603202401
8     145620202401
9      32190202401
10     32188202401
11    145624202401
12    145626202401
13     32186202401
14     32189202401
15    145604202401
16    144117202401
17     32171202401
18     32187202401
19    145612202401
Name: id, dtype: int64

### Diferencia estructural detectada en 2024

Durante la comparación de los datasets anuales se detectó que el archivo
correspondiente a 2024 contiene una variable adicional denominada `id`.

La variable no presenta valores faltantes y posee un valor único para cada
uno de los 983.551 registros del archivo. Sin embargo, no está presente en
los demás años y no es necesaria para identificar los pozos, ya que el
dataset dispone de `idpozo` y de las variables temporales `anio` y `mes`.

Por este motivo, para integrar los datasets se conservarán únicamente las
38 variables comunes al período 2021-2026, excluyendo `id` del archivo 2024.

In [39]:
columnas_auditoria = [
    "anio",
    "mes",
    "idpozo",
    "cuenca"
]

In [40]:
resumen_anual = []

for anio_archivo, archivo in archivos_anuales.items():
    # Se conservan únicamente las columnas comunes a todos los años.
    # Esto excluye la variable adicional "id" presente en 2024. 
    df_temp = pd.read_csv(
        archivo,
        usecols=columnas_auditoria
    )

    df_temp_cuyana = df_temp[
        df_temp["cuenca"] == "CUYANA"
    ]

    resumen_anual.append({
        "archivo": anio_archivo,
        "filas_totales": len(df_temp),
        "registros_cuyana": len(df_temp_cuyana),
        "pozos_cuyana": df_temp_cuyana["idpozo"].nunique(),
        "meses": df_temp["mes"].nunique(),
        "anio_min": df_temp["anio"].min(),
        "anio_max": df_temp["anio"].max()
    })

    del df_temp
    del df_temp_cuyana

resumen_anual = pd.DataFrame(resumen_anual)

resumen_anual

,archivo,filas_totales,registros_cuyana,pozos_cuyana,meses,anio_min,anio_max
0,2021,961451,43937,3673,12,2021,2021
1,2022,972394,44068,3683,12,2022,2022
2,2023,974971,44066,3673,12,2023,2023
3,2024,983551,44111,3676,12,2024,2024
4,2025,991844,44163,3685,12,2025,2025
5,2026,592297,25895,3690,8,2026,2026


### Interpretación de la compatibilidad temporal

Los datasets anuales presentan una estructura temporal consistente para
el período analizado.

Entre 2021 y 2025, cada archivo contiene información correspondiente a
los 12 meses del año. En la Cuenca Cuyana se observan aproximadamente
3.670 a 3.690 pozos únicos por año y alrededor de 44.000 registros
mensuales anuales.

El archivo correspondiente a 2026 contiene información de 8 meses, por
lo que se trata de un período parcial. Esta diferencia deberá considerarse
en los análisis temporales para evitar comparaciones directas entre 2026
y años completos.

En conjunto, los resultados indican que los datasets anuales son
compatibles para construir una serie histórica de producción de la
Cuenca Cuyana para el período 2021-2026.

El archivo 2024 contiene una variable adicional denominada `id`.
Para mantener una estructura homogénea se utilizarán únicamente las
38 variables comunes a todos los años.

## Construcción del dataset histórico de la Cuenca Cuyana

Una vez comprobada la compatibilidad de los archivos anuales, se
integrarán los registros correspondientes al período 2021-2026.

Para reducir el uso de memoria, cada archivo será cargado individualmente,
se conservarán únicamente las 38 variables comunes y se filtrarán los
registros correspondientes a la Cuenca Cuyana antes de realizar la
concatenación.

In [41]:
datos_cuyana = []

for anio, archivo in archivos_anuales.items():

    df_temp = pd.read_csv(
        archivo,
        usecols=columnas_referencia
    )
    # Se filtra Cuyana antes de concatenar para reducir
    # considerablemente el uso de memoria.
    df_temp = df_temp[
        df_temp["cuenca"] == "CUYANA"
    ].copy()

    datos_cuyana.append(df_temp)

    print(f"{anio}: {len(df_temp)} registros de Cuyana")

    del df_temp

C:\Users\milib\AppData\Local\Temp\ipykernel_20620\3955037375.py:5: DtypeWarning: Columns (0: sub_tipo_recurso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(


2021: 43937 registros de Cuyana


C:\Users\milib\AppData\Local\Temp\ipykernel_20620\3955037375.py:5: DtypeWarning: Columns (0: observaciones, 1: sub_tipo_recurso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(


2022: 44068 registros de Cuyana


C:\Users\milib\AppData\Local\Temp\ipykernel_20620\3955037375.py:5: DtypeWarning: Columns (0: observaciones, 1: sub_tipo_recurso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(


...

2024: 44111 registros de Cuyana


C:\Users\milib\AppData\Local\Temp\ipykernel_20620\3955037375.py:5: DtypeWarning: Columns (0: sub_tipo_recurso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(


2025: 44163 registros de Cuyana


C:\Users\milib\AppData\Local\Temp\ipykernel_20620\3955037375.py:5: DtypeWarning: Columns (0: sub_tipo_recurso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(


2026: 25895 registros de Cuyana


In [42]:
df_cuyana = pd.concat(
    datos_cuyana,
    ignore_index=True
)

print("Filas:", df_cuyana.shape[0])
print("Columnas:", df_cuyana.shape[1])

Filas: 246240
Columnas: 38


In [43]:
print("Pozos únicos:", df_cuyana["idpozo"].nunique())

print(
    "Período:",
    df_cuyana["anio"].min(),
    "-",
    df_cuyana["anio"].max()
)

print(
    "Duplicados pozo-año-mes:",
    df_cuyana.duplicated(
        subset=["idpozo", "anio", "mes"]
    ).sum()
)

Pozos únicos: 3702
Período: 2021 - 2026
Duplicados pozo-año-mes: 0


### Resultado de la integración

La integración de los datasets anuales permitió construir un dataset
histórico de la Cuenca Cuyana con 246.240 registros y 38 variables,
correspondientes al período 2021-2026.

En total se identificaron 3.702 pozos únicos.

No se encontraron registros duplicados para la combinación `idpozo`,
`anio` y `mes`, lo que confirma que cada observación representa un
registro mensual único para cada pozo.

El año 2026 contiene información parcial de 8 meses, mientras que
2021-2025 cuentan con los 12 meses del año.

In [44]:
padron = pd.read_csv(
    RUTA_PADRONES +
    "padrn-de-pozos-de-captulo-iv-con-fecha-de-primera-produccin.csv"
)

In [45]:
print("Filas:", padron.shape[0])
print("Columnas:", padron.shape[1])

padron.head()

Filas: 86197
Columnas: 3


,idpozo,anio,mes
0,212,2006,1
1,213,2006,1
2,214,2006,1
3,215,2006,1
4,216,2006,1


In [46]:
padron.info()

<class 'pandas.DataFrame'>
RangeIndex: 86197 entries, 0 to 86196
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   idpozo  86197 non-null  int64
 1   anio    86197 non-null  int64
 2   mes     86197 non-null  int64
dtypes: int64(3)
memory usage: 2.0 MB


In [47]:
padron.columns.tolist()

['idpozo', 'anio', 'mes']

In [48]:
print("Pozos registrados:", padron["idpozo"].nunique())

print(
    "IDs duplicados:",
    padron["idpozo"].duplicated().sum()
)

Pozos registrados: 86197
IDs duplicados: 0


In [49]:
padron.isnull().sum()

idpozo    0
anio      0
mes       0
dtype: int64

In [50]:
print(
    "Año mínimo de primera producción:",
    padron["anio"].min()
)

print(
    "Año máximo de primera producción:",
    padron["anio"].max()
)

print(
    "Mes mínimo:",
    padron["mes"].min()
)

print(
    "Mes máximo:",
    padron["mes"].max()
)

Año mínimo de primera producción: 2006
Año máximo de primera producción: 2026
Mes mínimo: 1
Mes máximo: 11


### Resultado de la auditoría del padrón

El padrón contiene 86.197 pozos y tres variables: `idpozo`, `anio` y
`mes`, correspondientes a la identificación del pozo y al año y mes
de su primera producción.

No se encontraron valores faltantes ni identificadores de pozo
duplicados, por lo que cada `idpozo` posee una única fecha de primera
producción registrada.

Las fechas de primera producción se encuentran entre 2006 y 2026.
Por lo tanto, el padrón resulta adecuado para complementar los
microdatos históricos y determinar la antigüedad productiva de los
pozos analizados.

In [51]:
pozos_cuyana = set(df_cuyana["idpozo"].unique())
pozos_padron = set(padron["idpozo"].unique())

pozos_con_fecha = pozos_cuyana.intersection(pozos_padron)
pozos_sin_fecha = pozos_cuyana - pozos_padron

print("Pozos únicos de Cuyana:", len(pozos_cuyana))
print("Pozos encontrados en el padrón:", len(pozos_con_fecha))
print("Pozos no encontrados en el padrón:", len(pozos_sin_fecha))

Pozos únicos de Cuyana: 3702
Pozos encontrados en el padrón: 3702
Pozos no encontrados en el padrón: 0


In [52]:
cobertura = len(pozos_con_fecha) / len(pozos_cuyana) * 100

print(f"Cobertura del padrón: {cobertura:.2f}%")

Cobertura del padrón: 100.00%


### Cobertura del padrón sobre la Cuenca Cuyana

Se compararon los identificadores de los pozos presentes en el dataset
histórico de la Cuenca Cuyana con los registrados en el padrón de primera
producción.

Los 3.702 pozos únicos identificados en el período 2021-2026 se encuentran
presentes en el padrón, obteniéndose una cobertura del 100%.

Este resultado permite asociar a todos los pozos analizados con su año y
mes de primera producción, haciendo posible estudiar su antigüedad
productiva y evaluar posteriormente los tiempos de maduración o ramp-up.

In [53]:
padron_cuyana = padron[
    padron["idpozo"].isin(pozos_cuyana)
].copy()

padron_cuyana.shape

(3702, 3)

In [54]:
pozos_por_inicio = (
    padron_cuyana
    .groupby("anio")["idpozo"]
    .count()
)

pozos_por_inicio

anio
2006    3290
2007      13
2008      16
2009       6
2010       5
2011      53
2013      11
2014      40
2015      54
2016      46
2017      52
2018      38
2019      27
2020       9
2021      13
2022      10
2023       1
2024       4
2025       9
2026       5
Name: idpozo, dtype: int64

In [55]:
pozos_rampup = padron_cuyana[
    padron_cuyana["anio"].between(2021, 2026)
].copy()

print(
    "Pozos que comenzaron entre 2021 y 2026:",
    len(pozos_rampup)
)

print(
    "Porcentaje sobre los pozos de Cuyana:",
    f"{len(pozos_rampup) / len(padron_cuyana) * 100:.2f}%"
)

Pozos que comenzaron entre 2021 y 2026: 42
Porcentaje sobre los pozos de Cuyana: 1.13%


### Viabilidad del análisis de ramp-up

El análisis del padrón muestra que solamente 42 de los 3.702 pozos
identificados en la Cuenca Cuyana comenzaron su producción entre 2021
y 2026, lo que representa aproximadamente el 1,13% del total.

La cantidad de pozos nuevos es reducida y, además, aquellos que comenzaron
en los años más recientes disponen de una ventana temporal limitada para
observar su evolución hasta alcanzar un eventual pico de producción.

Por este motivo, el ramp-up puede conservarse como una línea de análisis
exploratoria, pero su viabilidad como eje principal del proyecto deberá
evaluarse considerando la cantidad de meses de seguimiento disponible
para estos pozos.

In [56]:
padron_cuyana[
    padron_cuyana["anio"] == 2006
]["mes"].value_counts().sort_index()

mes
1    3290
Name: count, dtype: int64

### Particularidad detectada en las fechas de primera producción

Al analizar la distribución histórica del padrón se observó que 3.290
de los 3.702 pozos de la Cuenca Cuyana presentan enero de 2006 como
fecha de primera producción.

La concentración de todos estos registros en un mismo mes y año sugiere
la existencia de una particularidad asociada al inicio o disponibilidad
del registro histórico.

Por este motivo, estos casos no serán interpretados automáticamente como
pozos cuya primera producción real ocurrió en enero de 2006. Esta
limitación deberá considerarse al utilizar la antigüedad productiva como
variable de análisis.

In [57]:
ids_rampup = pozos_rampup["idpozo"]

seguimiento_rampup = (
    df_cuyana[
        df_cuyana["idpozo"].isin(ids_rampup)
    ]
    .groupby("idpozo")
    .size()
)

seguimiento_rampup.describe()

count    42.000000
mean     38.714286
std      22.798466
min       4.000000
25%      12.750000
50%      49.000000
75%      61.000000
max      65.000000
dtype: float64

In [58]:
seguimiento_rampup.sort_values()

idpozo
167194     4
167276     4
167193     4
167123     5
167079     6
166900     9
166730    12
166741    12
166740    12
166738    12
166739    12
166508    15
166516    15
166518    15
165041    30
164958    31
164959    31
164960    31
164094    41
163584    47
163412    49
163411    49
163410    49
163409    49
163225    51
162999    54
162998    54
163000    54
163062    54
162466    61
162463    61
162467    61
162464    61
162465    61
162468    61
162329    63
162331    63
162330    63
162191    65
162192    65
162189    65
162190    65
dtype: int64

### Evaluación final de la viabilidad del ramp-up

Se identificaron 42 pozos cuya primera producción ocurrió entre 2021
y 2026, equivalentes al 1,13% de los pozos analizados.

Estos pozos presentan una media de aproximadamente 39 meses de
seguimiento y una mediana de 49 meses. Algunos casos cuentan con hasta
65 meses de información, mientras que los pozos más recientes disponen
de solamente 4 meses.

Por lo tanto, existe información temporal suficiente para realizar un
análisis exploratorio del ramp-up en parte de estos pozos. Sin embargo,
la reducida cantidad de casos limita su utilización como eje principal
del proyecto.

El ramp-up se conservará como una línea complementaria de análisis.

In [59]:
from pathlib import Path

RUTA_PROCESADOS = Path("05_datos_procesados")
RUTA_PROCESADOS.mkdir(exist_ok=True)

In [60]:
df_cuyana.to_csv(
    RUTA_PROCESADOS / "produccion_pozos_cuenca_cuyana_2021_2026.csv",
    index=False
)

In [61]:
from pathlib import Path

archivo_salida = (
    RUTA_PROCESADOS /
    "produccion_pozos_cuenca_cuyana_2021_2026.csv"
)

print("Archivo creado:", archivo_salida.exists())
print(
    "Tamaño:",
    round(archivo_salida.stat().st_size / 1024**2, 2),
    "MB"
)

Archivo creado: True
Tamaño: 73.13 MB


In [62]:
df_prueba = pd.read_csv(archivo_salida)

print(df_prueba.shape)
df_prueba.head()

(246240, 38)


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,...,idareayacimiento,areayacimiento,cuenca,provincia,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data
0,YPF,2021,1,162019,0.00,0.00,235.47,0.0,0.0,0.0,...,UGA,UGARTECHE,CUYANA,Mendoza,CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,NaN,2021-01-31
1,YPF,2021,1,160838,91.23,3.23,10.23,0.0,0.0,0.0,...,UGA,UGARTECHE,CUYANA,Mendoza,CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,NaN,2021-01-31
2,YPF,2021,1,160267,32.58,1.20,83.11,0.0,0.0,0.0,...,UGA,UGARTECHE,CUYANA,Mendoza,CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,NaN,2021-01-31
3,YPF,2021,1,160265,843.53,30.47,313.61,0.0,0.0,0.0,...,MVER,MESA VERDE,CUYANA,Mendoza,CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,NaN,2021-01-31
4,YPF,2021,1,160142,37.81,2.10,189.83,0.0,0.0,0.0,...,LVEC,LA VENTANA CENTRAL,CUYANA,Mendoza,CONVENCIONAL,Sin Proyecto,SERVICIO,INYECTOR DE AGUA,NaN,2021-01-31


## Conclusiones preliminares

La auditoría permitió validar los microdatos de producción como fuente
principal para el proyecto.

Se construyó una base histórica de la Cuenca Cuyana para 2021-2026 con
246.240 registros, 38 variables y 3.702 pozos únicos. No se detectaron
duplicados para la combinación `idpozo`, `anio` y `mes`.

El padrón de primera producción presenta una cobertura del 100% sobre
los pozos analizados.

La evaluación inicial del ramp-up identificó 42 pozos cuya primera
producción ocurrió entre 2021 y 2026, equivalentes al 1,13% del total.
Aunque algunos presentan suficiente seguimiento temporal, el reducido
número de casos sugiere evaluar esta temática como análisis complementario
en lugar de asumirla directamente como eje principal.

### Próximos pasos

- Auditar los datasets de contexto macroeconómico y operativo.
- Analizar la evolución y el sostenimiento de la producción en Cuyana.
- Evaluar posibles indicadores de declinación productiva.
- Determinar junto con el grupo la pregunta principal del proyecto.
- Conservar el análisis de ramp-up como posible línea complementaria.